# Паттерн 9: Self-Reflection — генератор + критик в цикле

Паттерн Self-Reflection — один из самых мощных паттернов коллаборации в мультиагентных системах. Два агента работают в цикле итеративного улучшения: генератор (writer) создаёт текст, а критик (critic) оценивает его по набору критериев. Если текст не проходит проверку — критик возвращает конкретные замечания, и генератор дорабатывает результат. Цикл продолжается до одобрения критиком или до исчерпания лимита итераций.

```mermaid
graph LR
    START((START)) --> writer(["🔹 Writer<br/><i>создаёт черновик</i>"])
    writer --> critic(["🔹 Critic<br/><i>оценивает и критикует</i>"])
    critic -->|APPROVED| END((END))
    critic -->|needs work| writer

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px

    class START,END terminal
    class writer,critic worker
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние рефлексии

Состояние `ReflectionState` хранит текущий черновик, критику от предыдущей итерации, счётчик итераций и предел. Поле `iteration` увеличивается с каждым проходом генератора, а `max_iterations` служит обязательной страховкой от бесконечного цикла — без неё слишком требовательный критик, который никогда не говорит APPROVED, зациклит граф до `recursion_limit`.

In [3]:
llm = get_llm()

class ReflectionState(TypedDict):
    task: str
    draft: str
    critique: str
    iteration: int
    max_iterations: int

## Агент-генератор (Writer)

Генератор получает задачу и критику от предыдущей итерации. На первой итерации критики ещё нет — агент пишет текст с нуля. На последующих — дорабатывает черновик с учётом конкретных замечаний критика. Такое разделение промптов позволяет генератору фокусироваться: сначала на создании, потом на точечном улучшении.

In [4]:
def writer_node(state: ReflectionState) -> dict:
    """
    Генератор: пишет или улучшает текст.
    На первой итерации — создаёт с нуля.
    На последующих — дорабатывает с учётом критики.
    """
    iteration = state.get("iteration", 0)

    if iteration == 0:
        # Первая итерация — пишем с нуля
        prompt = f"Напиши короткий информативный текст (4-5 предложений) на тему: {state['task']}"
    else:
        # Последующие — улучшаем на основе критики
        prompt = (
            f"Улучши текст с учётом замечаний критика.\n\n"
            f"Текущий текст:\n{state['draft']}\n\n"
            f"Замечания критика:\n{state['critique']}\n\n"
            f"Верни улучшенную версию текста."
        )

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — талантливый писатель. Пиши ясно, информативно, "
            "с конкретными фактами и примерами. Структурируй текст."
        )),
        HumanMessage(content=prompt)
    ])

    new_iteration = iteration + 1
    print(f"  [Писатель, итерация {new_iteration}] Текст: {response.content[:80]}...")
    return {"draft": response.content, "iteration": new_iteration}

## Агент-критик (Critic)

Критик оценивает текст по четырём критериям: ясность, фактическая точность, увлекательность и полнота. Если текст проходит проверку по всем критериям — критик начинает ответ словом APPROVED. Это слово-маркер, по которому условное ребро графа решает, завершить цикл или отправить текст на доработку. Если нужны улучшения — критик формулирует 2-3 конкретных замечания.

In [5]:
def critic_node(state: ReflectionState) -> dict:
    """
    Критик: оценивает текст по критериям качества.
    Возвращает APPROVED (если текст хорош) или конкретные замечания.
    """
    response = llm.invoke([
        SystemMessage(content="""Ты — строгий, но справедливый критик.
Оцени текст по четырём критериям:
1. Ясность — понятен ли текст читателю
2. Фактическая точность — нет ли выдуманных утверждений
3. Увлекательность — интересно ли читать
4. Полнота — раскрыта ли тема

Если текст хорош по всем критериям — начни ответ со слова APPROVED.
Если нужны улучшения — напиши 2-3 конкретных замечания с указанием, что исправить."""),
        HumanMessage(content=f"Текст на оценку:\n{state['draft']}")
    ])

    content = response.content

    if "APPROVED" in content.upper():
        print(f"  [Критик, итерация {state['iteration']}] ОДОБРЕНО!")
        return {"critique": content}

    print(f"  [Критик, итерация {state['iteration']}] Замечания: {content[:80]}...")
    return {"critique": content}

## Условное ребро: продолжить или завершить

Функция маршрутизации `should_continue` проверяет два условия выхода из цикла: одобрение критиком (слово APPROVED в ответе) или исчерпание лимита итераций. Второе условие — обязательная страховка. Без неё слишком строгий критик зациклит граф до `recursion_limit`, что приведёт к ошибке и потере всех промежуточных результатов.

In [6]:
def should_continue(state: ReflectionState) -> str:
    """
    Условное ребро после критика:
    - APPROVED → завершение
    - Достигнут max_iterations → завершение
    - Иначе → ещё одна итерация писателя
    """
    if "APPROVED" in state.get("critique", "").upper():
        return "end"
    if state.get("iteration", 0) >= state.get("max_iterations", 3):
        print(f"  [Система] Достигнут лимит итераций ({state['max_iterations']})")
        return "end"
    return "continue"

## Сборка графа

Граф состоит из двух узлов и трёх рёбер. Безусловные рёбра: START -> writer и writer -> critic. Условное ребро из critic ведёт либо обратно к writer (если нужна доработка), либо к END (если текст одобрен или исчерпан лимит). Условное ребро с маппингом `{"continue": "writer", "end": END}` явно задаёт все возможные переходы.

In [7]:
graph = StateGraph(ReflectionState)

graph.add_node("writer", writer_node)
graph.add_node("critic", critic_node)

graph.add_edge(START, "writer")
graph.add_edge("writer", "critic")
graph.add_conditional_edges("critic", should_continue, {
    "continue": "writer",
    "end": END,
})

app = graph.compile()

## Запуск

Запускаем граф с задачей и лимитом в 3 итерации. На каждом шаге в консоль выводится прогресс: что написал генератор и что ответил критик. Типичный сценарий: первая итерация создаёт черновик, критик находит замечания, вторая итерация дорабатывает текст, критик одобряет — цикл завершается за 2 прохода.

In [8]:
result = app.invoke({
    "task": "Почему мультиагентные системы — главный тренд в AI-разработке 2025 года",
    "draft": "",
    "critique": "",
    "iteration": 0,
    "max_iterations": 3,
})

print(f"\n  Итого итераций: {result['iteration']}")
print(f"  Финальный текст: {result['draft'][:200]}...")
if "APPROVED" in result.get("critique", "").upper():
    print("  Статус: ОДОБРЕНО критиком")
else:
    print("  Статус: Завершено по лимиту итераций")

  [Писатель, итерация 1] Текст: Мультиагентные системы стали главным трендом AI-разработки 2025 года, потому что...


  [Критик, итерация 1] Замечания: Есть несколько проблем:

1. **Фактическая точность** — утверждение, что мультиаг...


  [Писатель, итерация 2] Текст: Мультиагентные системы — один из заметных трендов AI-разработки 2025 года. Их по...


  [Критик, итерация 2] Замечания: Текст в целом хороший, но до идеала не хватает нескольких уточнений.

1. **Факти...


  [Писатель, итерация 3] Текст: Мультиагентные системы сегодня **считаются одним из заметных трендов в AI-разраб...


  [Критик, итерация 3] ОДОБРЕНО!

  Итого итераций: 3
  Финальный текст: Мультиагентные системы сегодня **считаются одним из заметных трендов в AI-разработке**: интерес к ним растёт потому, что несколько ИИ-агентов могут делить между собой задачи, обмениваться результатами...
  Статус: ОДОБРЕНО критиком


## Итоги

Паттерн Self-Reflection реализует цикл обратной связи между двумя специализированными агентами. Ключевые элементы реализации:

- **Два агента с разными ролями** — генератор фокусируется на создании контента, критик — на оценке качества. Разделение ролей даёт лучший результат, чем один агент, пытающийся одновременно писать и оценивать.
- **Условное ребро** — маршрутизация по маркеру APPROVED в ответе критика. Простой строковый маркер надёжнее, чем парсинг JSON-структуры от LLM.
- **Лимит итераций** — обязательная страховка от зацикливания. Без `max_iterations` слишком строгий критик может бесконечно требовать доработок.
- **Итеративное улучшение** — каждая следующая версия текста учитывает конкретные замечания критика, что обеспечивает направленное улучшение вместо случайных переписываний.